# Wine Quality: tuned Random Forest comparison

**Question:** how does splitting by wine type (red / white) affect Random Forest quality regression, and which chemical features matter most?

**Scenarios:** red only, white only, pooled with `wine_type`, and pooled without `wine_type`.

**Scheme:** cleaned data → mean baseline → nested cross-validation (5 outer folds, 3 inner folds for parameter search) → separately tuned 80/20 hold-out models → permutation importance.

Each Random Forest scenario gets its own reproducible parameter search. Outer validation rows and hold-out test rows never enter the corresponding parameter search. “Best” means lowest mean inner-CV RMSE among sampled candidates, not a guaranteed global optimum.

The cleaned file is produced by `wine_quality_eda.ipynb`; duplicates are removed before splitting to prevent identical rows from leaking across train and validation sets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, KFold, RandomizedSearchCV, train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, mean_squared_error, mean_absolute_error, r2_score)

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print("numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

## Evaluation helper (provided)

In [ ]:
def evaluate_classification_metrics(model, X_valid, y_valid, title="Classification Report"):
    """
    Universal function for evaluating models (including regressors)
    with classification metrics.
    """
    # 1. Make predictions
    predictions = model.predict(X_valid)

    # 2. If the model returns fractional numbers (regression), round them
    predictions_rounded = np.round(predictions)

    # 3. Compute the standard metrics
    # zero_division=0 prevents warnings if the model never predicted a given class
    acc = accuracy_score(y_valid, predictions_rounded)
    prec = precision_score(y_valid, predictions_rounded, average='macro', zero_division=0)
    rec = recall_score(y_valid, predictions_rounded, average='macro', zero_division=0)
    f1 = f1_score(y_valid, predictions_rounded, average='macro', zero_division=0)

    print(f"--- {title} ---")
    print(f"Accuracy:  {acc:.4f} (Share of exact hits)")
    print(f"Precision: {prec:.4f} (Macro)")
    print(f"Recall:    {rec:.4f} (Macro)")
    print(f"F1-Score:  {f1:.4f} (Macro)\n")

    # 4. Classification of the error size (our own metric)
    exact_matches = np.sum(predictions_rounded == y_valid)
    off_by_one = np.sum(np.abs(predictions_rounded - y_valid) == 1)
    total = len(y_valid)

    print("--- Error distribution ---")
    print(f"Exact match:            {exact_matches/total*100:.2f}%")
    print(f"Off by 1 point:         {off_by_one/total*100:.2f}%")
    print(f"Gross error (>1 point): {(total - exact_matches - off_by_one)/total*100:.2f}%\n")

    # 5. Confusion matrix visualisation
    cm = confusion_matrix(y_valid, predictions_rounded)

    # Get the unique classes (scores) that are actually present in the validation set
    classes = np.unique(np.concatenate([y_valid, predictions_rounded]))

    plt.figure(figsize=(8, 6))
    # annot=True shows the numbers, cmap='Blues' sets the blue colour map
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes)

    plt.title(f'Confusion Matrix: {title}')
    plt.ylabel('Actual score')
    plt.xlabel('Predicted score')
    plt.tight_layout()
    plt.show()

## 1. Loading the cleaned data

In [ ]:
# The cleaned dataset is produced by wine_quality_eda.ipynb: duplicate rows are already removed
# (so identical rows cannot end up in both train and test) and there are no missing values.
CLEAN_PATH = "dataset/wine_clean.csv"

raw = pd.read_csv(CLEAN_PATH)

# Sanity checks: the experiment relies on these properties of the cleaned data
assert raw.isna().sum().sum() == 0, "unexpected missing values"
assert not raw.duplicated().any(), "unexpected duplicate rows"

# wine_type is stored as text in the CSV; the models need a number: 0 = red, 1 = white.
# quality_category (low / medium / high) is not used in this notebook, so it is dropped.
df = (raw.assign(wine_type=raw["wine_type"].map({"red": 0, "white": 1}))
         .drop(columns="quality_category"))
assert df["wine_type"].notna().all(), "unknown wine_type value"

print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Combined dataset:", df.shape)
display(df["wine_type"].map({0: "red", 1: "white"}).value_counts().rename("rows").to_frame())

In [ ]:
# Target distribution after cleaning
dist = pd.crosstab(df["quality"], df["wine_type"].map({0: "red", 1: "white"}))
display(dist)
dist.plot(kind="bar", figsize=(7, 3.5), title="Distribution of quality (no duplicates)")
plt.tight_layout(); plt.show()

## 2. Experiment setup and parameter search

* Model: `RandomForestRegressor` inside a `Pipeline`; no scaling is needed.
* Search: 12 sampled configurations per fit, 3-fold shuffled inner CV, scored by negative RMSE. Search covers tree count, depth, feature sampling, minimum split/leaf sizes, and bootstrap sampling. Increase `SEARCH_ITERATIONS` for a broader search.
* Inner CV uses `KFold` because this is regression and extreme quality scores have very few examples. The outer folds remain stratified by `wine_type + quality` and identical for every scenario.
* Nested CV: tune only on each outer training fold, refit its best model, then predict the untouched outer validation fold. Inner best scores are tuning diagnostics; outer predictions estimate generalisation.
* Hold-out: run a fresh search for each scenario on its 80% training subset, then evaluate its refitted best model on the shared 20% test split.
* Baseline: `DummyRegressor(strategy="mean")`, without tuning.
* Search jobs run in parallel; each forest uses one worker to avoid nested CPU oversubscription.
* Metrics: RMSE, MAE, R², rounded accuracy, and the share within ±1 point.

**Fair comparison:** pooled and separate models are evaluated on the same red and white validation rows. The hold-out is a secondary evaluation on overlapping data with the full-data outer CV, not an additional independent dataset.

This is substantially slower than fitting fixed forests: the default configuration performs 24 searches (20 outer-fold searches plus 4 hold-out searches).

In [ ]:
FEATURES = [c for c in df.columns if c not in ("quality", "wine_type")]
FEATURES_WITH_TYPE = FEATURES + ["wine_type"]

SEARCH_ITERATIONS = 12
INNER_SPLITS = 3
SEARCH_JOBS = -1
PARAM_DISTRIBUTIONS = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 10, 20, 30],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": [1.0, 0.7, "sqrt"],
    "rf__bootstrap": [True, False],
}

def make_rf():
    return Pipeline([("rf", RandomForestRegressor(n_jobs=1, random_state=SEED))])

def tune_rf(X_train, y_train):
    search = RandomizedSearchCV(
        estimator=make_rf(),
        param_distributions=PARAM_DISTRIBUTIONS,
        n_iter=SEARCH_ITERATIONS,
        scoring="neg_root_mean_squared_error",
        cv=KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED),
        refit=True,
        n_jobs=SEARCH_JOBS,
        random_state=SEED,
        return_train_score=False,
        error_score="raise",
    )
    return search.fit(X_train, y_train)

def make_dummy():
    return Pipeline([("dummy", DummyRegressor(strategy="mean"))])

def reg_metrics(y_true, y_pred):
    r = np.round(y_pred)
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, r),
        "Within±1": np.mean(np.abs(r - y_true) <= 1),
    }

## 3. Nested cross-validation: tune within each outer training fold

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
strat = df["wine_type"].astype(str) + "_" + df["quality"].astype(str)
folds = list(skf.split(df, strat))

y = df["quality"].to_numpy()
is_red, is_white = (df["wine_type"] == 0).to_numpy(), (df["wine_type"] == 1).to_numpy()

scenarios = {
    "baseline (mean)":          dict(feats=FEATURES,           per_type=False, factory=make_dummy),
    "red only":                 dict(feats=FEATURES,           per_type=True,  factory=make_rf, only=0),
    "white only":               dict(feats=FEATURES,           per_type=True,  factory=make_rf, only=1),
    "combined + wine_type":     dict(feats=FEATURES_WITH_TYPE, per_type=False, factory=make_rf),
    "combined, no wine_type":   dict(feats=FEATURES,           per_type=False, factory=make_rf),
}

cv_search_rows = []
oof = {}   # out-of-fold predictions: name -> array of length len(df) (NaN where the scenario is not applicable)
for name, cfg in scenarios.items():
    pred = np.full(len(df), np.nan)
    for fold_no, (tr, te) in enumerate(folds, start=1):
        if "only" in cfg:                       # separate model: train and test only on its own type
            tr = tr[df["wine_type"].to_numpy()[tr] == cfg["only"]]
            te = te[df["wine_type"].to_numpy()[te] == cfg["only"]]
        X_train = df.iloc[tr][cfg["feats"]]
        if name == "baseline (mean)":
            m = make_dummy().fit(X_train, y[tr])
        else:
            search = tune_rf(X_train, y[tr])
            m = search.best_estimator_
            cv_search_rows.append({
                "scenario": name,
                "outer_fold": fold_no,
                "best inner CV RMSE": -search.best_score_,
                **search.best_params_,
            })
            print(f"{name}, outer fold {fold_no}: inner CV RMSE = {-search.best_score_:.4f}")
        pred[te] = m.predict(df.iloc[te][cfg["feats"]])
    oof[name] = pred
cv_search_results = pd.DataFrame(cv_search_rows)
display(cv_search_results)
print("Done: outer-fold predictions use independently tuned models.")

In [ ]:
# Combine 'red only' + 'white only' into the "separate approach" (each wine is scored by the model of its own type)
oof["separate (red+white models)"] = np.where(is_red, oof["red only"], oof["white only"])

def table(mask, names):
    rows = {n: reg_metrics(y[mask], oof[n][mask]) for n in names}
    return pd.DataFrame(rows).T.round(4)

names_cmp = ["baseline (mean)", "separate (red+white models)", "combined + wine_type", "combined, no wine_type"]

print("=== All wines (red + white) ===")
res_all = table(np.ones(len(df), bool), names_cmp); display(res_all)
print("=== Red only ===")
res_red = table(is_red, names_cmp); display(res_red)
print("=== White only ===")
res_white = table(is_white, names_cmp); display(res_white)

In [ ]:
# The same, but "a separate model on its own dataset" in the form it is usually compared (metrics on its own test data)
own = pd.DataFrame({
    "red model → red":     reg_metrics(y[is_red],   oof["red only"][is_red]),
    "white model → white": reg_metrics(y[is_white], oof["white only"][is_white]),
}).T.round(4)
display(own)

### Stability across folds
A single RMSE number can be a coincidence; we look at the spread across folds (±std) of RMSE separately for red and white.

In [ ]:
rows = []
for name in ["separate (red+white models)", "combined + wine_type", "combined, no wine_type", "baseline (mean)"]:
    for tname, mask in [("red", is_red), ("white", is_white)]:
        vals = []
        for tr, te in folds:
            te_m = te[mask[te]]
            vals.append(np.sqrt(mean_squared_error(y[te_m], oof[name][te_m])))
        rows.append((name, tname, np.mean(vals), np.std(vals)))
fold_df = pd.DataFrame(rows, columns=["scenario", "subset", "RMSE mean", "RMSE std"]).round(4)
display(fold_df.pivot(index="scenario", columns="subset", values=["RMSE mean", "RMSE std"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (title, tbl) in zip(axes, [("All", res_all), ("Red", res_red), ("White", res_white)]):
    tbl["RMSE"].plot(kind="barh", ax=ax, color=sns.color_palette("deep")[:len(tbl)])
    ax.set_title(f"RMSE (CV): {title}"); ax.invert_yaxis()
    for i, v in enumerate(tbl["RMSE"]): ax.text(v, i, f" {v:.3f}", va="center")
plt.tight_layout(); plt.show()

## 4. Hold-out (80/20): fresh training-only searches

One shared stratified split; the same test rows are used for all applicable scenarios. Separate models are tuned on their own wine type. Every search refits its best estimator on that scenario's entire training subset. Only these refitted estimators are used below for evaluation and permutation importance.

The table reports the best parameters and inner-CV RMSE; the subsequent hold-out metrics evaluate unseen test rows.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED,
                                     stratify=df["wine_type"].astype(str) + "_" + df["quality"].astype(str))
tr_red,  te_red  = train_df[train_df.wine_type == 0], test_df[test_df.wine_type == 0]
tr_white, te_white = train_df[train_df.wine_type == 1], test_df[test_df.wine_type == 1]
print(len(train_df), "train /", len(test_df), "test")

holdout_specs = {
    "red only": (tr_red, FEATURES),
    "white only": (tr_white, FEATURES),
    "combined + wine_type": (train_df, FEATURES_WITH_TYPE),
    "combined, no wine_type": (train_df, FEATURES),
}
holdout_searches = {}
holdout_search_rows = []
for name, (train_subset, feats) in holdout_specs.items():
    search = tune_rf(train_subset[feats], train_subset["quality"])
    holdout_searches[name] = search
    holdout_search_rows.append({
        "scenario": name,
        "best inner CV RMSE": -search.best_score_,
        **search.best_params_,
    })
    print(f"{name}: best inner CV RMSE = {-search.best_score_:.4f}")

holdout_search_results = pd.DataFrame(holdout_search_rows).set_index("scenario")
display(holdout_search_results)

# Show all candidates, ranked within each scenario, for inspecting the search.
holdout_candidates = pd.concat([
    pd.DataFrame(search.cv_results_).assign(scenario=name)
    for name, search in holdout_searches.items()
], ignore_index=True)
holdout_candidates["mean CV RMSE"] = -holdout_candidates["mean_test_score"]
display(holdout_candidates.sort_values(["scenario", "rank_test_score"])[
    ["scenario", "rank_test_score", "mean CV RMSE", "std_test_score", "params"]
])

rf_red = holdout_searches["red only"].best_estimator_
rf_white = holdout_searches["white only"].best_estimator_
rf_comb = holdout_searches["combined + wine_type"].best_estimator_
rf_nowt = holdout_searches["combined, no wine_type"].best_estimator_

In [ ]:
evaluate_classification_metrics(rf_red, te_red[FEATURES], te_red["quality"], title="RED model on RED test")

In [ ]:
evaluate_classification_metrics(rf_white, te_white[FEATURES], te_white["quality"], title="WHITE model on WHITE test")

In [ ]:
evaluate_classification_metrics(rf_comb, test_df[FEATURES_WITH_TYPE], test_df["quality"], title="COMBINED + wine_type (all test)")

In [ ]:
evaluate_classification_metrics(rf_nowt, test_df[FEATURES], test_df["quality"], title="COMBINED, no wine_type (all test)")

In [ ]:
# Hold-out summary by subset (identical test rows for all)
def holdout_row(model_fn, sub, feats_fn):
    return reg_metrics(sub["quality"].to_numpy(), model_fn(sub[feats_fn]))

hold = pd.DataFrame({
    "separate → red":        holdout_row(rf_red.predict,  te_red,   FEATURES),
    "combined+type → red":   holdout_row(rf_comb.predict, te_red,   FEATURES_WITH_TYPE),
    "combined no type → red":holdout_row(rf_nowt.predict, te_red,   FEATURES),
    "separate → white":        holdout_row(rf_white.predict, te_white, FEATURES),
    "combined+type → white":   holdout_row(rf_comb.predict,  te_white, FEATURES_WITH_TYPE),
    "combined no type → white":holdout_row(rf_nowt.predict,  te_white, FEATURES),
}).T.round(4)
display(hold)

## 5. How much worse does the model get without `wine_type`
Difference of the pooled model without type relative to the pooled model with type (CV, out-of-fold). Positive ΔRMSE = worse.

In [ ]:
d = pd.DataFrame({
    "RMSE with type":    [reg_metrics(y[m], oof["combined + wine_type"][m])["RMSE"] for m in (np.ones(len(df), bool), is_red, is_white)],
    "RMSE without type": [reg_metrics(y[m], oof["combined, no wine_type"][m])["RMSE"] for m in (np.ones(len(df), bool), is_red, is_white)],
}, index=["all", "red", "white"])
d["ΔRMSE"] = d["RMSE without type"] - d["RMSE with type"]
d["Δ%"] = d["ΔRMSE"] / d["RMSE with type"] * 100
display(d.round(4))

## 6. Permutation importance (on the hold-out test set)
Computed on the test part (not on train), `n_repeats=20`. We show feature importance for the pooled model with `wine_type`, for the separate models, and for the model without the type.

In [ ]:
def perm_imp(model, X, y_):
    r = permutation_importance(model, X, y_, n_repeats=20, random_state=SEED,
                               scoring="neg_root_mean_squared_error", n_jobs=-1)
    return pd.Series(r.importances_mean, index=X.columns)

pi = pd.DataFrame({
    "red model":        perm_imp(rf_red,   te_red[FEATURES],   te_red["quality"]),
    "white model":      perm_imp(rf_white, te_white[FEATURES], te_white["quality"]),
    "combined + type":  perm_imp(rf_comb,  test_df[FEATURES_WITH_TYPE], test_df["quality"]),
    "combined no type": perm_imp(rf_nowt,  test_df[FEATURES],           test_df["quality"]),
})
pi = pi.sort_values("combined + type", ascending=False)
display(pi.round(4))

ax = pi.plot(kind="barh", figsize=(9, 7), title="Permutation importance (RMSE increase when the feature is shuffled)")
ax.invert_yaxis(); plt.tight_layout(); plt.show()

## 7. Error analysis: where the model fails

In [ ]:
# MAE for each true score (CV, pooled model with wine_type)
err = pd.DataFrame({"quality": y, "abs_err": np.abs(y - oof["combined + wine_type"]),
                    "type": df["wine_type"].map({0: "red", 1: "white"})})
by_q = err.groupby("quality")["abs_err"].agg(["mean", "count"]).round(3)
display(by_q)
by_q["mean"].plot(kind="bar", figsize=(6, 3), title="MAE by true score (combined + wine_type)")
plt.tight_layout(); plt.show()

## 8. Conclusions and limitations

Run all cells to obtain the tuned comparison. Previous fixed-parameter outputs and numerical conclusions have been cleared because they do not describe these searched models.

Use `cv_search_results` to inspect the parameters selected inside every outer fold. Use `res_all`, `res_red`, `res_white`, and `fold_df` to compare generalisation on identical validation rows. The inner search scores are used for parameter selection and should not be reported as unbiased model performance.

Use `holdout_search_results` and `holdout_candidates` to inspect the final searches. The classification reports, `hold` table, and permutation importance use their refitted best estimators. Compare the effect of removing `wine_type` using `d`; avoid assuming that tuning must improve every outer fold or hold-out score.

**Limitations:**
* Randomized search explores a finite sample of the stated parameter space; it does not guarantee globally optimal parameters.
* Hyperparameters are selected independently per scenario and per outer fold; there is no single parameter set underlying all out-of-fold predictions.
* Rare extreme quality scores and subjective labels limit performance; rounded macro classification metrics can be unstable.
* Fold-to-fold RMSE spread describes variability and is not a formal significance test.
* Full-row duplicate removal cannot distinguish different wines with identical measurements.
* Correlated features can dilute permutation importance.
* The hold-out and full-data nested CV overlap, so they are secondary views of the same dataset.
* README figures and results from the earlier fixed-parameter experiment are historical; rerun this notebook before reporting tuned results.

**Reproducibility:** all splits, searches, and forests use `SEED = 42`; library versions are printed in the first cell. Run `jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 wine_rf_comparison.ipynb`. Runtime depends on available CPU cores and `SEARCH_ITERATIONS`; nested search is substantially slower than the previous fixed-model notebook.